# box-array-to-tensor-with-recipe composite — cx12: boxing preserves requires_grad across the three-gate rule

> Composite procedural drill from [Delta Drills](https://delta-drills.vercel.app).
> Exercises 2 atoms together: `box-array-to-tensor-with-recipe`, `requires-grad-propagation`
> Running the final beacon reports progress against all 2 subtopics.

**Why composite drills.** Single-atom drills test atomic skills in isolation. Composite drills test the COMPOSITION — how atoms wire together in real ARENA code. Passing this drill demonstrates you can apply the atoms jointly, not just individually.

## Setup

In [ ]:
import numpy as np
import torch as t
from torch import Tensor
import einops
from einops import rearrange, reduce, repeat

t.manual_seed(0)
np.random.seed(0)

## Connect to Delta Drills

Paste your Delta Drills auth token below. Beacon will report progress against ALL atoms exercised by this composite.

In [ ]:
# === Delta Drills auth (composite) ===
DD_TOKEN = ""  # paste token, then run
DD_PRIMARY_ATOM = "box-array-to-tensor-with-recipe"
DD_ATOM_IDS = ["box-array-to-tensor-with-recipe", "requires-grad-propagation"]
DD_SUBTOPICS = ["Backprop: Box array as Tensor + recipe", "Backprop: requires_grad propagation"]
DD_BACKEND_URL = "https://delta-drills-backend.fly.dev"

_dd_passed = set()

## Boxing as the carrier for the three-gate requires_grad rule

The wrapper computes `requires_grad` as the AND of three gates:

```python
requires_grad = (
    grad_tracking_enabled
    and is_differentiable
    and any(isinstance(a, MiniTensor) and a.requires_grad for a in args)
)
```

Whatever bool falls out of that AND, **boxing carries it through unchanged** onto the output's `requires_grad` AND uses it to decide Recipe attachment. Boxing is the single point where the three-gate decision becomes the output Tensor's state.

### Composite Exercise — boxing preserves requires_grad across the three-gate rule

**Atoms exercised together**: `box-array-to-tensor-with-recipe`, `requires-grad-propagation`

Implement `cx12_box_with_three_gate(out_raw, fwd_fn, args, raw_args, kwargs, is_differentiable, grad_tracking_enabled)`.

Compute the three-gate AND:
```
requires_grad = (
    grad_tracking_enabled
    AND is_differentiable
    AND any(isinstance(a, MiniTensor) and a.requires_grad for a in args)
)
```

Then box: `out = MiniTensor(out_raw, requires_grad=requires_grad)`. If `requires_grad`, attach `out.recipe = Recipe(fwd_fn, raw_args, kwargs, parents)` where `parents` is built from `args` (argidx-preserving, MiniTensor-only).

The boxed output's `requires_grad` must EXACTLY equal the three-gate value (no recomputation, no drift).

In [ ]:
# Fill in the function below, then run this cell. The test asserts the composition is correct.

def cx12_box_with_three_gate(out_raw, fwd_fn, args, raw_args, kwargs, is_differentiable, grad_tracking_enabled):
    """Box raw output. requires_grad = AND of all three gates."""
    raise NotImplementedError

def _test_cx12():
    T1 = MiniTensor(t.tensor([1.0]), requires_grad=True)
    T0 = MiniTensor(t.tensor([1.0]), requires_grad=False)
    raw_out = t.tensor([1.0])

    # --- truth table: all-True → rg=True + Recipe ---
    out = cx12_box_with_three_gate(raw_out, t.log, (T1,), (T1.array,), {}, True, True)
    assert out.array is raw_out, 'box must not copy storage'
    assert out.requires_grad is True
    assert out.recipe is not None
    assert out.recipe.func is t.log
    assert out.recipe.parents == {0: T1}

    # --- any single gate False → rg=False + no Recipe ---
    # gate 1 off: global toggle
    out = cx12_box_with_three_gate(raw_out, t.log, (T1,), (T1.array,), {}, True, False)
    assert out.requires_grad is False
    assert out.recipe is None
    # gate 2 off: non-differentiable op
    out = cx12_box_with_three_gate(raw_out, t.equal, (T1,), (T1.array,), {}, False, True)
    assert out.requires_grad is False
    assert out.recipe is None
    # gate 3 off: no tracked input
    out = cx12_box_with_three_gate(raw_out, t.log, (T0,), (T0.array,), {}, True, True)
    assert out.requires_grad is False
    assert out.recipe is None

    # --- mixed inputs: scalar must not block rg, raw torch.Tensor must not contribute ---
    out = cx12_box_with_three_gate(raw_out, t.multiply, (T1, 3.0), (T1.array, 3.0), {}, True, True)
    assert out.requires_grad is True, 'float at arg-1 must not veto'
    assert out.recipe.parents == {0: T1}, 'scalar excluded from parents'
    raw_pass = t.tensor([7.0])
    out = cx12_box_with_three_gate(raw_out, t.multiply, (raw_pass, T1), (raw_pass, T1.array), {}, True, True)
    assert out.requires_grad is True
    assert out.recipe.parents == {1: T1}, 'raw torch.Tensor not a MiniTensor → skipped'

    # --- AttributeError defense: must not ask non-Tensors for .requires_grad ---
    class Sneaky:
        def __getattr__(self, name):
            if name == 'requires_grad':
                raise AttributeError('do not touch')
            raise AttributeError(name)
    out = cx12_box_with_three_gate(raw_out, t.log, (Sneaky(),), (object(),), {}, True, True)
    assert out.requires_grad is False
    assert out.recipe is None
    out = cx12_box_with_three_gate(raw_out, t.add, (Sneaky(), T1), (object(), T1.array), {}, True, True)
    assert out.requires_grad is True, 'Sneaky must be skipped, not crash'
    assert out.recipe.parents == {1: T1}

    # --- empty args → False ---
    out = cx12_box_with_three_gate(raw_out, t.tensor, (), (), {}, True, True)
    assert out.requires_grad is False
    assert out.recipe is None
    _dd_passed.add('cx12')

_test_cx12()

<details><summary>Show solution — cx12</summary>

```python
from dataclasses import dataclass, field
from typing import Any, Callable, Optional

grad_tracking_enabled = True

@dataclass
class Recipe:
    func: Optional[Callable] = None
    args: tuple = ()
    kwargs: dict = field(default_factory=dict)
    parents: dict = field(default_factory=dict)

class MiniTensor:
    def __init__(self, array, requires_grad: bool = False, recipe=None):
        self.array = array
        self.requires_grad = requires_grad
        self.recipe = recipe

def cx12_box_with_three_gate(out_raw, fwd_fn, args, raw_args, kwargs, is_differentiable, grad_tracking_enabled):
    requires_grad = bool(
        grad_tracking_enabled
        and is_differentiable
        and any(
            isinstance(a, MiniTensor) and a.requires_grad
            for a in args
        )
    )
    out = MiniTensor(out_raw, requires_grad=requires_grad)
    if requires_grad:
        parents = {
            idx: a
            for idx, a in enumerate(args)
            if isinstance(a, MiniTensor)
        }
        out.recipe = Recipe(fwd_fn, raw_args, kwargs, parents)
    return out
```

**Boxing is the carrier — not the computer — of the three gates.** Each gate has its own home: global toggle lives in the module, `is_differentiable` lives on the op registration, input-rg lives on the arguments. The boxer ANDs them and writes the result to `out.requires_grad`. The `bool(...)` cast is defensive — `and` over non-bools (e.g. a `0` slipping in) could give a falsy non-bool that fails strict `is True` / `is False` tests later.
</details>

## Report completion

Run the cell below to send progress to Delta Drills. The beacon fires once and reports all 2 subtopics together.

In [ ]:
# === Delta Drills completion beacon (composite — fires for ALL atoms) ===
import urllib.request as _dd_req, json as _dd_json

_DD_REQUIRED = {'cx12'}

def report_completion():
    missing = _DD_REQUIRED - _dd_passed
    if missing:
        print(f"[Delta Drills] {sorted(missing)} not yet passing — fix the cell above, then re-run this one.")
        return
    if not DD_TOKEN:
        print('[Delta Drills] DD_TOKEN is empty — completion not reported.')
        return
    body = _dd_json.dumps({
        'exercise_title': f'composite-drill:{DD_PRIMARY_ATOM}:cx12',
        'subtopics': ["Backprop: Box array as Tensor + recipe", "Backprop: requires_grad propagation"],
        'feedback': 'somewhat',
        'correct': True,
    }).encode('utf-8')
    req = _dd_req.Request(
        f'{DD_BACKEND_URL}/api/practice/arena-rating',
        data=body,
        headers={
            'Content-Type': 'application/json',
            'Authorization': f'Bearer {DD_TOKEN}',
        },
        method='POST',
    )
    try:
        with _dd_req.urlopen(req, timeout=5) as r:
            resp = _dd_json.loads(r.read())
        print(f'[Delta Drills] reported composite (atoms={DD_ATOM_IDS})')
        print(f'[Delta Drills] EWMA updated: {resp}')
    except Exception as e:
        print(f'[Delta Drills] beacon failed: {e}')

report_completion()